In [ ]:
import os
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'
import json

In [ ]:
import datasets
from datasets import load_dataset

### Build case study viewer data.

In [ ]:
ds = load_dataset('thoughtworks/psychometric_personas_temp')

In [ ]:
sjt = load_dataset('thoughtworks/psychometric_sjts')['train']

In [ ]:
ds.column_names

In [ ]:
sjt

In [ ]:
ds_train = ds['train']

In [ ]:
ds_train

In [ ]:
targets = ['a32089a1-1740-417b-87f2-266d3db53b61', 'de216cca-84ff-4724-806a-ef96d530b450']

In [ ]:
ds_filtered = ds_train.filter(lambda x: x['uuid'] in targets)

In [ ]:
ds_filtered_list = ds_filtered.to_list()
names_dict = {item['uuid']: item['name'] for item in ds_filtered_list}

In [ ]:
names_dict

In [ ]:
hexaco_filtered = ds_filtered

In [ ]:
case_study_path = '/workspace/shreyans_workspace/psychometrics_for_LLMs/llm_psychometrics/experiment_results/case_study_data'

In [ ]:
hexaco_data = '/workspace/shreyans_workspace/psychometrics_for_LLMs/llm_psychometrics/experiment_results/case_study_data/huggingface_hexaco_answers_gpt-4_1-mini.json'
sjt_data = '/workspace/shreyans_workspace/psychometrics_for_LLMs/llm_psychometrics/experiment_results/case_study_data/huggingface_sjt_answers_gpt-4_1-mini.json'

In [ ]:
with open(hexaco_data, 'r') as f_in:
    case_study_hexaco = json.load(f_in)

with open(sjt_data, 'r') as f_in:
    case_study_sjt = json.load(f_in)

In [ ]:
cs_hexaco = {key: value for key, value in case_study_hexaco.items() if key in targets}
cs_sjt = {key: value for key, value in case_study_sjt.items() if key in targets}

In [ ]:
# cs_hexaco

In [ ]:
cs_hexaco

In [ ]:
with open('cs_hexaco.json' , 'w') as f_out:
    json.dump(cs_hexaco, f_out)

In [ ]:
qs_hashes = cs_sjt[targets[0]]['config']['question_hashes']

In [ ]:
sjt_filtered = sjt.filter(lambda x: x['hash_id'] in qs_hashes)

In [ ]:
sjt_list = sjt_filtered.to_list()

In [ ]:
sjt_dict = {item['hash_id']: item for item in sjt_list if item['hash_id'] in qs_hashes}

In [ ]:
len(sjt_dict)

In [ ]:
raw_options = [
    'honesty-humility', 'emotionality', 'extraversion', 'agreeableness',
    'conscientiousness', 'openness to experience'
]
raw_scores = {target: {key: 0 for key in raw_options} for target in targets}

In [ ]:
sjt_results = {target: [] for target in targets}

In [ ]:
raw_scores = {target: {key: 0 for key in raw_options} for target in targets}
sjt_results = {target: [] for target in targets}
for target in cs_sjt:
    #print(answers)
    answers = cs_sjt[target]['answers'][0]
    answer_indices = cs_sjt[target]['config']['answer_index']
    hashes = cs_sjt[target]['config']['question_hashes']
    #print(answer_index)

    for answer, answer_index, curr_hash in zip(answers, answer_indices, hashes):
        sjt_option = answer_index[int(answer) - 1]
        raw_scores[target][raw_options[sjt_option]] += 1
        sjt_question = sjt_dict[curr_hash]

        sjt_results[target].append({"hash": curr_hash, "option": raw_options[sjt_option], "question": sjt_question})

sjt_results_final = {names_dict[id]: result for id, result in sjt_results.items()}
#sjt_results

In [ ]:
# sjt_results_final

In [ ]:
with open('case_study_sjt_results.json', 'w') as f_out:
    json.dump(sjt_results_final, f_out)

In [ ]:
# work now with sjt_filtered, hexaco_filtered, hexaco_data, sjt_data

In [ ]:
sjt_filtered['corrected_sjt']

In [ ]:
sjt

In [ ]:
names_dict

### Diversity metrics last time.

In [ ]:
import os
os.chdir("/workspace/psychometrics_for_LLMs/llm_psychometrics/src/")

In [ ]:
from evals.diversity_metrics import DiversityMetrics

In [ ]:
from datasets import load_dataset
ds = load_dataset("thoughtworks/psychometric_personas")

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
texts = ds["train"].to_dict()["persona_string"]

In [ ]:
texts[4]

In [ ]:
dm = DiversityMetrics(texts)

In [ ]:
all_metrics = DiversityMetrics.compute_all()

In [ ]:
! kill -9  2734216

In [ ]:
! nvidia-smi